# Lab 18: Model Evaluation — The Curve, and What It Cannot Tell You
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 67 min Core + 20 min Write it yourself + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Draw ROC and precision-recall curves and compare what each shows under imbalance
- Show that a single summary number hides the operating point
- Compare two models with a paired bootstrap, and explain why one AUC cannot rank them
- Convert an evaluation into a decision using stated costs
- Write a small function that gives one point on a classifier's ROC curve

**Dataset:** Simulated imbalanced classification outcomes (measured: 11.8% positives among the test rows)

**Prerequisites:** Lab 17 (`predict_proba`, the confusion matrix, precision and recall), and the labs before it.

**How this lab works:** Parts 1-4 are GUIDED: run each cell as it is, then answer the questions.
"Write it yourself" is required: a short function you write from a blank cell. The Extension is
optional, and the Challenge is a take-home.

**Plan on about 1¾ hours** (1 hour 45 minutes) for the required work: setup, Parts 1-4, Write it
yourself, and the README and submission.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Write it yourself (required),** after Part 4: `roc_point()`, a function that gives back one point on the ROC curve: the false positive rate and the true positive rate at any threshold. About four lines from a blank cell. The check cell below it must print "Passed"; it then uses your function at Part 3's cost-minimising threshold.
> - The answers to the interpretation questions after each part, in text cells: two for each Part.
>
> **You run and read:** everything else. Setup and Parts 1-4 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The CHECK cell after Write it yourself is different: you only run it and read its message; you do not need to follow its code.
>
> **The Extension** (optional) and **the Challenge** (take-home) have no starter code: you write them in new cells, from the task. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

`GradientBoostingClassifier` is the second model in Part 2; the rest is Lab 17's toolkit.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the seed
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import metrics       # the curves, the scores and the confusion matrix

RANDOM_STATE = 42     # the seed for the simulated data, the split, the boosted model and the bootstrap

print("Setup complete.")

## Part 1: Two Curves, One Model (GUIDED — 20 min)

Step 2 simulates 8,000 cases of a rare outcome, so the truth is known. Each case has five features,
drawn from a standard normal: `rng.normal(0, 1, (n, 5))` makes a table of `n` rows and 5 columns.
`Xm @ np.array([...])` is a matrix product: for each row it multiplies the five features by the five
weights and adds them up. With the intercept −2.9, that is the true log-odds of a positive; the
fourth feature has weight 0, so it is noise. As in Lab 17, `1 / (1 + np.exp(-lin))` turns the
log-odds into a probability, and `rng.binomial(1, ...)` flips one coin per case.

The split is Lab 17's: 30% of the rows are kept for testing, and `stratify=ym` gives both sets the
same share of positives. `yb.mean()` is the share of 1s among the test rows: the prevalence.

In [ ]:
# GUIDED — run as-is
# Step 2: 8,000 simulated cases, split into training and test rows
rng = np.random.default_rng(seed=RANDOM_STATE)     # seeded here, so rerunning gives the same cases
n = 8_000
Xm = rng.normal(0, 1, (n, 5))
lin = -2.9 + Xm @ np.array([1.2, -0.9, 0.7, 0.0, 0.4])     # the true log-odds
ym = rng.binomial(1, 1 / (1 + np.exp(-lin)))

Xa, Xb, ya, yb = train_test_split(Xm, ym, test_size=0.3,
                                  random_state=RANDOM_STATE, stratify=ym)
print(f"prevalence = {yb.mean():.1%}")
print(f"positives among the {len(yb):,} test rows: {yb.sum()}")

**Expected output:** `prevalence = 11.8%`, and 283 positives among the 2,400 test rows. `Xa` and `ya` are the 5,600 training rows; `Xb` and `yb`
are the 2,400 test rows, and everything in this lab is scored on them.

Step 3 fits Lab 17's pipeline (scale, then logistic regression) on the training rows, and keeps each
test row's probability of a positive: column 1 of `predict_proba`. It prints nothing.

In [ ]:
# GUIDED — run as-is
# Step 3: fit on the training rows, and keep each test row's probability of a positive
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xa, ya)
pr = model.predict_proba(Xb)[:, 1]

Lab 17 scored one threshold at a time. Step 4 tries every threshold at once and draws two curves.

- The **ROC curve** plots the **true positive rate** (the share of actual positives that are
  flagged; Lab 17 called it recall) against the **false positive rate** (the share of actual
  negatives that are flagged). `metrics.roc_curve(y_true, prob)` gives back three arrays, one entry
  per threshold it tries: the false positive rates, the true positive rates, and the thresholds.
- The **precision-recall curve** plots precision against recall.
  `metrics.precision_recall_curve(y_true, prob)` gives back precision, recall and the thresholds. A
  name of just `_` is the usual way to say "a value we do not need".

Each curve has a one-number summary, printed in its chart title. `roc_auc_score` is the area under
the ROC curve (AUC); it is also the probability that a randomly chosen positive gets a higher
predicted probability than a randomly chosen negative (here 0.8468, rounded to 0.847 in the title).
`average_precision_score` (AP) is the precision averaged over the recall levels.

`plt.subplots(1, 2, ...)` makes one figure with two charts side by side, `axes[0]` and `axes[1]`.
The dashed lines are what a classifier that assigns random scores would draw.

In [ ]:
# GUIDED — run as-is
# Step 4: the ROC curve and the precision-recall curve for the same probabilities
fpr, tpr, _ = metrics.roc_curve(yb, pr)
prec, rec, _ = metrics.precision_recall_curve(yb, pr)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(fpr, tpr, color="navy", lw=2)
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("false positive rate")
axes[0].set_ylabel("true positive rate")
axes[0].set_title(f"ROC — AUC {metrics.roc_auc_score(yb, pr):.3f}")

axes[1].plot(rec, prec, color="firebrick", lw=2)
axes[1].axhline(yb.mean(), ls="--", color="gray", label="prevalence")
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title(f"Precision-Recall — AP {metrics.average_precision_score(yb, pr):.3f}")
axes[1].legend()
plt.tight_layout()
plt.show()

**Expected output:** two charts. The ROC chart's title reads `AUC 0.847`; the precision-recall
chart's reads `AP 0.465`, with its dashed line at 0.118.

### Interpretation Questions

1. AUC is well above 0.5 while average precision is much lower. Explain why the two disagree at low prevalence.
2. A classifier that assigns random scores would trace the diagonal on the ROC chart, but the dashed line at the prevalence on the precision-recall chart. Explain why its precision stays at the prevalence whatever its recall.

*Your answers here:*

1.
2.

## Part 2: Two Models, One Test Set (GUIDED — 22 min)

Step 5 fits a second model and puts its AUC and AP beside the first model's. Step 7 then asks how
much the gap between the two would change on a different test set: the 2,400 test rows are one
sample, so every score computed on them has sampling error.

The second model is a **gradient-boosted** classifier: a sum of 120 small decision trees
(`n_estimators=120`), each two splits deep (`max_depth=2`), each fitted to the errors of the trees
before it. Topic 19 covers trees; here it is simply a flexible second model. Trees do not need
scaled features, so it has no pipeline.

In Step 5, `for name, p in [("logistic", pr), ("boosted", pr2)]` walks through a list of pairs and
unpacks each pair into `name` and `p`.

In [ ]:
# GUIDED — run as-is
# Step 5: a second model, and the two summary numbers for each
m2 = GradientBoostingClassifier(n_estimators=120, max_depth=2,
                                random_state=RANDOM_STATE).fit(Xa, ya)
pr2 = m2.predict_proba(Xb)[:, 1]

print(f"{'model':<14}{'ROC AUC':>10}{'AP':>10}")
print("-" * 34)
for name, p in [("logistic", pr), ("boosted", pr2)]:
    print(f"{name:<14}{metrics.roc_auc_score(yb, p):>10.4f}"
          f"{metrics.average_precision_score(yb, p):>10.4f}")

**Expected output:** logistic `0.8468` and `0.4651`; boosted `0.8431` and `0.4529`.

AUC and AP each summarise every threshold at once, while a business acts at one threshold. Step 6
reads each model's precision at four levels of recall. `precision_at_recall` finds the point on the
precision-recall curve whose recall is closest to the target: `np.abs(rc - target)` is each point's
distance from the target, and `np.argmin` gives the position of the smallest one.

In [ ]:
# GUIDED — run as-is
# Step 6: precision at four levels of recall, for each model
def precision_at_recall(y_true, prob, target):
    """Precision at the point on the PR curve closest to `target` recall."""
    pc, rc, _ = metrics.precision_recall_curve(y_true, prob)
    return pc[np.argmin(np.abs(rc - target))]

print(f"{'recall target':>14}{'logistic prec':>16}{'boosted prec':>15}")
print("-" * 46)
for target in [0.2, 0.4, 0.6, 0.8]:
    print(f"{target:>14.1f}{precision_at_recall(yb, pr, target):>16.3f}"
          f"{precision_at_recall(yb, pr2, target):>15.3f}")

**Expected output:** four rows. Logistic: 0.626, 0.509, 0.384, 0.279. Boosted: 0.564, 0.500, 0.402,
0.273.

Step 7 asks how much two of those gaps would move on another test set, with a **paired bootstrap**:
resample the test set with replacement, re-score *both* models on the same resample, and record the
difference. Pairing matters: an easy resample makes both models look good, and we only care about
the gap.

- `boot.integers(0, len(yb), len(yb))` draws 2,400 row positions from 0 to 2,399, with repeats, and
  `yb[idx]` and `pr[idx]` pick those rows.
- A resample with no positives, or nothing but positives, cannot be scored, so `continue` skips to
  the next one; with 283 positives among the test rows it does not happen here.
- The `difference` column is the gap on the full test set, as in Steps 5 and 6.
  `np.percentile(d, [2.5, 97.5])` gives the values below which 2.5% and 97.5% of the 500 resampled
  gaps fall: the 95% interval. `:+.4f` prints a number with its sign.

In [ ]:
# GUIDED — run as-is
# Step 7: a paired bootstrap of the gap, logistic minus boosted
B = 500
boot = np.random.default_rng(RANDOM_STATE)     # its own generator, so Step 2's data is untouched
d_auc, d_prec = [], []
for _ in range(B):
    idx = boot.integers(0, len(yb), len(yb))
    if yb[idx].sum() in (0, len(idx)):        # a resample with one class only
        continue
    d_auc.append(metrics.roc_auc_score(yb[idx], pr[idx])
                 - metrics.roc_auc_score(yb[idx], pr2[idx]))
    d_prec.append(precision_at_recall(yb[idx], pr[idx], 0.6)
                  - precision_at_recall(yb[idx], pr2[idx], 0.6))
d_auc, d_prec = np.array(d_auc), np.array(d_prec)

# the gaps on the full test set, which the resamples put an interval around
gap_auc = metrics.roc_auc_score(yb, pr) - metrics.roc_auc_score(yb, pr2)
gap_prec = precision_at_recall(yb, pr, 0.6) - precision_at_recall(yb, pr2, 0.6)

print(f"{B} paired bootstrap resamples of the test set (logistic minus boosted):")
print(f"{'':<26}{'difference':>12}{'95% interval':>24}")
print("-" * 62)
for label, gap, d in [("ROC AUC", gap_auc, d_auc), ("precision at recall 0.6", gap_prec, d_prec)]:
    lo, hi = np.percentile(d, [2.5, 97.5])
    interval = f"[{lo:+.4f}, {hi:+.4f}]"
    print(f"{label:<26}{gap:>+12.4f}{interval:>24}")

**Expected output:** `ROC AUC` `+0.0038` with interval `[-0.0020, +0.0094]`, and
`precision at recall 0.6` `-0.0181` (Step 6's 0.384 − 0.402) with interval `[-0.0473, +0.0254]`.

### Interpretation Questions

1. Both bootstrap intervals contain zero. Write the one-sentence conclusion you would put in a report — one that is neither "the models are identical" nor "the logistic model is better."
2. Look at Step 6's table. The ranking appears to flip between recall 0.4 and recall 0.6. Given Step 7's interval on that difference, are you entitled to act on the flip? What would you need in order to become entitled to?

*Your answers here:*

1.
2.

## Part 3: From Curve to Decision (GUIDED — 13 min)

Lab 17 priced a lender's two errors and found the cheapest threshold. Step 8 does the same here,
with a missed positive (a false negative) costing ten times a false alarm (a false positive).

`expected_cost` flags every case whose probability is `t` or more, counts the errors with Lab 17's
`.ravel()` of the confusion matrix (true negatives, false positives, false negatives, true
positives), and spreads their cost over every test case. `cost_curve` repeats that at each of 200
thresholds from 0.01 to 0.9 (`np.linspace(0.01, 0.9, 200)`); Part 4 uses both functions again.
`np.argmin(costs)` is the position of the lowest cost, so `ts[...]` at that position is the
threshold. The last line prices Lab 17's closed form, FP_COST / (FP_COST + FN_COST), which assumes
calibrated probabilities.

In [ ]:
# GUIDED — run as-is
# Step 8: the expected cost at every threshold, and the cheapest one
FN_COST, FP_COST = 10.0, 1.0      # a missed positive costs ten times a false alarm
ts = np.linspace(0.01, 0.9, 200)

def expected_cost(y, prob, t, fn_cost, fp_cost=FP_COST):
    """Expected cost per case when every case with probability t or more is flagged."""
    yp = (prob >= t).astype(int)
    tn, fp, fn, tp = metrics.confusion_matrix(y, yp).ravel()
    return (fn * fn_cost + fp * fp_cost) / len(y)

def cost_curve(y, prob, fn_cost):
    """The expected cost per case at every threshold in ts."""
    costs = []
    for t in ts:
        costs.append(expected_cost(y, prob, t, fn_cost))
    return np.array(costs)

costs = cost_curve(yb, pr, FN_COST)
best_t = ts[np.argmin(costs)]
t_closed = FP_COST / (FP_COST + FN_COST)
print(f"cost-minimising threshold {best_t:.3f}, cost {costs.min():.4f}")
print(f"cost at the default 0.5   {expected_cost(yb, pr, 0.5, FN_COST):.4f}")
print(f"closed-form threshold     {t_closed:.3f}, cost {expected_cost(yb, pr, t_closed, FN_COST):.4f}")

**Expected output:** `0.068`, the cheapest of the 200 thresholds tried, at a cost of `0.4587` per case, `0.9437`
at the default 0.5, and the closed form's `0.091` at `0.4683`.

Step 9 draws the whole curve, with the cheapest threshold and the default marked.

In [ ]:
# GUIDED — run as-is
# Step 9: the cost curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ts, costs, color="navy", lw=2)
ax.axvline(best_t, ls="--", color="firebrick", label=f"optimum {best_t:.3f}")
ax.axvline(0.5, ls=":", color="gray", label="default 0.5")
ax.set_xlabel("threshold")
ax.set_ylabel("expected cost per case")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation Questions

1. How far is the cost-minimising threshold from 0.5, and from Lab 17's closed form, in threshold and in cost? What does the shape of Step 9's curve near its minimum mean for how precisely the business needs to state its cost ratio?
2. Chapter 18 says the missing ingredient is not statistical. Say what it is and who in an organisation supplies it.

*Your answers here:*

1.
2.

## Part 4: Your Operating Point (GUIDED — 12 min)

Part 3 fixed the cost ratio at 10:1. Step 10 lets it run from 2:1 to 50:1, and for each ratio finds
both models' cheapest threshold and its cost, with Step 8's `cost_curve`. A false alarm still costs
1, so the ratio is the cost of a miss.

In [ ]:
# GUIDED — run as-is
# Step 10: the cheapest threshold and its cost, for both models, at six cost ratios
print(f"{'FN:FP':>7}{'logistic t':>12}{'cost':>8}{'boosted t':>12}{'cost':>8}")
print("-" * 47)
for ratio in [2, 5, 10, 20, 30, 50]:
    c1 = cost_curve(yb, pr, ratio)
    c2 = cost_curve(yb, pr2, ratio)
    print(f"{ratio:>5}:1{ts[np.argmin(c1)]:>12.3f}{c1.min():>8.4f}"
          f"{ts[np.argmin(c2)]:>12.3f}{c2.min():>8.4f}")

**Expected output:** six rows. The 10:1 row repeats Step 8 for the logistic model (`0.068`,
`0.4587`), and gives `0.077` and `0.4733` for the boosted one. A threshold of 0.010 is the lowest
the grid tries.

### Interpretation Questions

1. At which cost ratios is each model the cheaper one, and does that change as the ratio rises? Given Part 2's intervals, would you act on it?
2. Your manager wants one "best model" shipped this week. What would you need before "which model is better" is even a well-posed question? Name one ground *other than* predictive score on which you could choose between these two models, and say why.

*Your answers here:*

1.
2.

---

## Write it yourself (required — 20 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** Step 4's `metrics.roc_curve` tries every threshold at once and gives back the whole
ROC curve. Write a function that gives back one point on it.

Write `roc_point(model, X, y, t)` that flags every row of `X` whose predicted probability of a
positive, from `model`, is `t` or more, and gives back two numbers: the false positive rate, then
the true positive rate, of those flags against the true labels `y`. That is the ROC chart's x, then
its y.

- It takes about four lines: the `def` line, a line that makes the 0/1 flags, a line for the counts,
  and the `return`. Lab 17's `precision_recall_at()` had the same shape; only the two shares change.
- Step 4 defines both rates, and Step 8 shows how to get the four counts.
- The probabilities come from `model` and `X`, not from `pr`. Step 3 shows which column of
  `predict_proba` holds the probability of a positive.
- `return a, b` gives back two values together. Give back the full numbers; round only when you print.
- If you compute a share by hand, Python may display it as `np.float64(0.37...)`. That is just a
  number.

Then run the check cell. It compares your function with Step 4's ROC curve at three thresholds, and
tries it on other rows, the other model, and a threshold equal to a row's own probability. `assert`
stops with its message when its condition is False; the message tells you what to fix.

In [ ]:
# WRITE IT YOURSELF (required): roc_point(model, X, y, t)

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
if "roc_point" not in globals():
    print("Not written yet: write roc_point() in the cell above and run it, then run this check.")
else:
    import copy, re

    # Step 3's model must still give Step 3's probabilities on Step 2's test rows
    for fitted, probs, where in [(model, pr, "Step 3's model"), (m2, pr2, "Step 5's m2")]:
        same = (np.shape(Xb) == (len(probs), 5) and np.shape(yb) == (len(probs),)
                and np.allclose(fitted.predict_proba(Xb)[:, 1], probs))
        assert same, (
            f"{where} no longer gives its Part 1-4 probabilities on Xb. Either it was refitted, perhaps "
            "by an earlier version of your function, or a cell redefined Xb or yb. Write only the "
            "function, rerun Parts 1-4, then this cell.")

    def call(mdl, X, y, t):
        """Run your roc_point on a copy of the model; if it goes wrong, say where to look."""
        m = copy.deepcopy(mdl)                    # a copy, so your function cannot change the original
        before = m.predict_proba(X)
        # try/except catches an error inside your function, so the message can name the cause
        try:
            out = roc_point(m, X, y, t)
        except Exception as e:
            hint = "Read the error above: it says what went wrong."
            sizes = [int(s) for s in re.findall(r"\d+", str(e))]   # the row counts in the message
            if any(s in str(e) for s in ["positional argument", "predict_proba", "Expected 2D array"]):
                hint = "roc_point takes four arguments, in this order: model, X, y, t."
            elif "inconsistent numbers of samples" in str(e) and sizes and min(sizes) <= 2:
                hint = "[1] or [:1] picks rows of predict_proba; you want a column, as in Step 3."
            elif "inconsistent numbers of samples" in str(e):
                hint = (f"It was handed {len(X):,} rows and mixed them with a different number. Does it "
                        "use its own X and y everywhere, or Step 3's pr and yb?")
            elif "multilabel" in str(e) or "multioutput" in str(e):
                hint = ("Its flags have two columns. predict_proba gives one column per class: "
                        "use only the probability of a positive.")
            # str(e)[:200] keeps the first 200 characters of the error message
            raise AssertionError(f"roc_point on {len(X):,} rows at t = {t:.4f} stopped with an "
                                 f"error:\n  {str(e)[:200]}\n{hint}") from None
        assert np.allclose(m.predict_proba(X), before), (
            "Your function refits the model; only use predict_proba. Rerun Parts 1-4, then this cell.")
        unchanged = (np.allclose(model.predict_proba(Xb)[:, 1], pr)
                     and np.allclose(m2.predict_proba(Xb)[:, 1], pr2))
        assert unchanged, ("Your function refits Step 3's model or Step 5's m2; only use predict_proba. "
                           "Rerun Parts 1-4, then this cell.")
        return two_numbers(out, len(X))

    def two_numbers(out, rows):
        """Check that out is two numbers, and hand them back as plain floats."""
        assert out is not None, "roc_point gave back nothing. Does it end with a return line?"
        assert not hasattr(out, "predict_proba"), (
            "roc_point gives back the model. Give back the false positive rate and the true positive rate.")
        assert np.size(out) != rows, (
            "roc_point gives back one value per row: the flags. Count them, and give back the two rates.")
        assert np.size(out) == 2, (f"roc_point gives back {np.size(out)} value(s). Give back two "
                                   "numbers: the false positive rate, then the true positive rate.")
        f, r = np.ravel(np.asarray(out, dtype=float))
        return f, r

    def curve_point(y, prob, target):
        """The point on the library's ROC curve whose threshold is closest to target."""
        c_fpr, c_tpr, c_t = metrics.roc_curve(y, prob)
        k = int(np.argmin(np.abs(c_t - target)))
        return float(c_t[k]), (c_fpr[k], c_tpr[k])

    t1, (F, T) = curve_point(yb, pr, 0.2)      # a point on Step 4's ROC curve, near t = 0.2
    f, r = call(model, Xb, yb, t1)

    assert not (np.isclose(f, 100 * F) or np.isclose(r, 100 * T)), (
        "Your numbers are percents. Give back shares between 0 and 1, as Step 4's curve does.")
    assert f <= 1 and r <= 1, (
        f"roc_point gives back {f:g} and {r:g}: counts, not rates. A rate is a share between 0 and 1; "
        "divide each count by the right total.")
    # round(v, places) keeps that many decimals; try 1 to 8 of them
    for places in [1, 2, 3, 4, 5, 6, 7, 8]:
        assert not (np.allclose((f, r), (round(F, places), round(T, places)), rtol=0, atol=1e-12)
                    and not np.allclose((f, r), (F, T), rtol=1e-9, atol=0)), (
            "Your numbers are rounded. Give back the full numbers; round only when you print.")
    assert not (np.isclose(f, T) and np.isclose(r, F)), (
        "Your first number is the true positive rate and your second the false positive rate. Give "
        "back the ROC chart's x first (the false positive rate), then its y.")
    assert not np.isclose(f, 1 - F), (
        "Your first number is the share of actual negatives that are NOT flagged. The false positive "
        "rate counts the actual negatives that ARE flagged.")
    # library scores of the right flags, to name the common mix-ups
    flags = (pr >= t1).astype(int)
    P = metrics.precision_score(yb, flags)               # share of flagged rows that are positives
    NPV = metrics.precision_score(1 - yb, 1 - flags)     # share of unflagged rows that are negatives
    assert not (np.isclose(f, P) and np.isclose(r, T)), (
        "You give back the precision and the recall, as Lab 17's precision_recall_at() did. The first "
        "number here is the false positive rate: a share of the actual negatives, not of the rows flagged.")
    assert not ((np.isclose(f, 1 - P) and np.isclose(r, NPV))
                or (np.isclose(f, 1 - NPV) and np.isclose(r, P))), (
        "Two of the four counts are swapped. Check the order of the arguments to confusion_matrix "
        "(true labels first) and the order .ravel() gives the counts in (Step 8).")
    assert not np.isclose(r, P), (
        "Your true positive rate is the precision: it divides by the rows flagged. Check the order of "
        "the arguments to confusion_matrix (true labels first), and which rows each rate divides by.")
    assert not np.isclose(f, 1 - P), (
        "Your first number divides by the rows flagged: it is the share of flagged rows that are "
        "negatives. The false positive rate is a share of the actual negatives.")
    assert not np.isclose(r, 1 - T), (
        "Your second number is the share of actual positives NOT flagged. The true positive rate "
        "counts the actual positives that ARE flagged.")
    # the library's ROC curve for the probability of a NEGATIVE, at the same threshold
    o_fpr, o_tpr, o_t = metrics.roc_curve(yb, model.predict_proba(Xb)[:, 0], drop_intermediate=False)
    k0 = np.where(o_t >= t1)[0][-1]
    assert not np.allclose((f, r), (o_fpr[k0], o_tpr[k0])), (
        "Your function flags rows by the probability of a NEGATIVE. Step 3 shows which column of "
        "predict_proba holds the probability of a positive.")

    # three thresholds that flag different numbers of rows should give three different points
    t2, _ = curve_point(yb, pr, 0.5)
    t3, _ = curve_point(yb, pr, best_t)
    points = [(f, r), call(model, Xb, yb, t2), call(model, Xb, yb, t3)]
    assert not (np.allclose(points[0], points[1]) and np.allclose(points[0], points[2])), (
        "roc_point gives the same point at every threshold. Does it use its t argument? "
        ".predict() always cuts at 0.5; make the flags from the probabilities and t.")

    # t1 is a row's own probability: at exactly that threshold, the row should be flagged
    assert np.allclose(call(model, Xb, yb, t1), call(model, Xb, yb, np.nextafter(t1, 0))), (
        "At a threshold equal to a row's own probability, your function does not flag that row. The "
        "task flags every row whose probability is t or more.")

    # other rows and another model, against the library's ROC curve for them
    for mdl, X, y, where, fix in [
            (model, Xa, ya, "the training rows", "its X and y arguments, or Step 3's pr and yb"),
            (m2, Xb, yb, "Step 5's boosted model", "its model argument, or Step 3's model")]:
        t_o, want = curve_point(y, mdl.predict_proba(X)[:, 1], 0.2)
        mine = call(mdl, X, y, t_o)
        if not np.allclose(mine, want, rtol=1e-9, atol=0):
            assert not np.allclose(mine, call(model, Xb, yb, t_o)), (
                f"roc_point gives the same answer for {where} as for Step 3's model on the test rows. "
                f"Does it use {fix}?")

    for target in [0.2, 0.5, best_t, 0.03]:
        t_k, want = curve_point(yb, pr, target)
        mine = call(model, Xb, yb, t_k)
        # round(v, places) keeps that many decimals; try 1 to 8 of them
        for places in [1, 2, 3, 4, 5, 6, 7, 8]:
            rounded = (round(want[0], places), round(want[1], places))
            assert not (np.allclose(mine, rounded, rtol=0, atol=1e-12)
                        and not np.allclose(mine, want, rtol=1e-9, atol=0)), (
                "Your numbers are rounded. Give back the full numbers; round only when you print.")
        assert np.allclose(mine, want, rtol=1e-9, atol=0), (
            f"roc_point(model, Xb, yb, {t_k:.4f}) gives {mine[0]:.6f} and {mine[1]:.6f}; Step 4's ROC "
            f"curve has {want[0]:.6f} and {want[1]:.6f} there. Check three things: the probabilities, "
            "the comparison with t, and which rows each rate divides by.")
    for mdl, X, y in [(model, Xa, ya), (m2, Xb, yb)]:
        t_o, want = curve_point(y, mdl.predict_proba(X)[:, 1], 0.2)
        assert np.allclose(call(mdl, X, y, t_o), want, rtol=1e-9, atol=0), (
            "roc_point matches Step 4's curve but not the ROC curve of other rows or another model. "
            "Does it use its own model, X and y everywhere?")

    f_b, r_b = call(model, Xb, yb, best_t)
    print("Passed. roc_point matches the library's ROC curve at every threshold tried, and uses the "
          "model, rows and threshold it is given.")
    print(f"At Part 3's cost-minimising threshold, {best_t:.3f}: false positive rate {f_b:.3f}, "
          f"true positive rate {r_b:.3f}")

---

## Extension (Optional — 15 min): Making the Difference Real

Part 2 ended with a gap you could not tell from zero. Here the outcome gets two extra terms, and you
compare the same two models on it. There is no starter code: write it in the cell below.

1. Rebuild Step 2's data with two terms the logistic model cannot represent, an interaction and a
   square. Use new names (`Xm2`, `ym2`, ...), so Parts 1-4 keep their data:
   ```python
   lin2 = (-2.9 + Xm2 @ np.array([1.2, -0.9, 0.7, 0.0, 0.4])
           + 1.5 * Xm2[:, 0] * Xm2[:, 1]
           + 1.2 * (Xm2[:, 2] ** 2 - 1))
   ```
   Everything else stays the same: the same n, the same split, the same two models.
2. Refit both models, and rerun Step 7's paired bootstrap on the new test rows.

**Your answer.** Report each model's AUC and the 95% interval on the difference, next to Step 7's.
In Part 2 you could not rank the models and here you can: did the algorithms change, or did
something else? What does that tell you about a benchmark table that reports scores without
intervals?

In [ ]:
# EXTENSION (optional) — write your code here

*Your answer here:*

---

## Challenge (Take-Home): The Metric That Chose Itself

There is no starter code: write your answer in the cell below and in new cells under it. Use the
Extension's data if you did it, and Part 1's if not.

1. Fit four models of differing complexity.
2. Rank them by accuracy, by ROC AUC, by average precision, and by expected cost at a 10:1 ratio
   (Step 8's functions).
3. Show where the rankings disagree.
4. An analyst free to choose the metric after seeing the results can justify almost any model.
   Propose the discipline that prevents it.

If you skip the Challenge, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated 8,000 cases of a rare outcome; [YOUR VALUE] of the
  test rows were positive
* Drew a ROC curve and a precision-recall curve for a logistic
  regression: ROC AUC [YOUR VALUE], average precision [YOUR VALUE]
* Compared it with a boosted-tree model (ROC AUC [YOUR VALUE]): the
  AUC difference on the full test set was [YOUR VALUE], and a paired
  bootstrap over 500 resamples gave a 95% interval of [YOUR VALUE]
* Built an expected-cost curve with a miss costing ten times a false
  alarm: cost-minimising threshold [YOUR VALUE] at [YOUR VALUE] per
  case, against [YOUR VALUE] at the default 0.5
* Swept the cost ratio from 2:1 to 50:1 for both models and checked
  whether the cheaper model changes
* Wrote a small function, roc_point(), that gives one point on the
  ROC curve; at my cost-minimising threshold the false positive rate
  was [YOUR VALUE] and the true positive rate [YOUR VALUE]
* (Only if I did the Extension) With a non-linear truth: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Model Evaluation: The Curve, and What It Cannot Tell You
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab18-model-evaluation`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab18-model-evaluation`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab18-model-evaluation`, branch `main`,
   commit message `Lab 18 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.